[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Open-Medica/open-medical-skills/blob/main/oms-models/notebooks/01_data_generation.ipynb)

# 01 - Synthetic Training Data Generation

**OMS Custom Embedding Model Pipeline - Step 1 of 6**

This notebook generates synthetic (query, tool-description) training pairs from three data sources:
- **49 OMS Skills** (physician-reviewed medical AI skills)
- **5 OMS Plugins** (full medical AI plugin packages)
- **1,995 ToolUniverse Tools** (Harvard/MIT medical tool catalog)

For each tool/skill, an LLM generates 10-20 diverse search queries across 8 query types:
direct functional, clinical scenario, research question, category-based, skill-specific,
comparative, negation, and multi-step.

**Output**: JSONL file with ~30,000+ (query, description) pairs for downstream model training.

---

## 1. Setup & Dependencies

Install all required packages. On Colab, these are not pre-installed.

In [ ]:
!pip install -q sentence-transformers google-generativeai anthropic openai pyyaml datasets tqdm requests

In [ ]:
import json
import os
import sys
import time
from collections import Counter
from pathlib import Path

import yaml
from tqdm.notebook import tqdm

print("All imports successful.")

## 2. Mount Google Drive (Optional)

Mount Google Drive to persist large output files across Colab sessions.
Skip this cell if running locally.

In [ ]:
# Uncomment to mount Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# OUTPUT_DIR = Path('/content/drive/MyDrive/oms-models/data/raw')

# Local output directory (default)
OUTPUT_DIR = Path('data/raw')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Output directory: {OUTPUT_DIR}")

## 3. Configure API Keys

Set up API keys for LLM backends. Gemini is recommended (free tier available).
You only need ONE backend configured -- Gemini is the default.

In Colab, use the Secrets panel (key icon in left sidebar) or set env vars directly.

In [ ]:
# Option 1: Set keys directly (replace with your actual keys)
# os.environ['GEMINI_API_KEY'] = 'your-gemini-api-key-here'
# os.environ['ANTHROPIC_API_KEY'] = 'your-anthropic-api-key-here'  # optional
# os.environ['OPENAI_API_KEY'] = 'your-openai-api-key-here'  # optional

# Option 2: Load from Colab Secrets
try:
    from google.colab import userdata
    os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')
    print("Loaded GEMINI_API_KEY from Colab Secrets")
    try:
        os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
        print("Loaded ANTHROPIC_API_KEY from Colab Secrets")
    except Exception:
        pass
    try:
        os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
        print("Loaded OPENAI_API_KEY from Colab Secrets")
    except Exception:
        pass
except ImportError:
    print("Not running in Colab. Using environment variables.")

# Verify at least one key is set
available_backends = []
if os.getenv('GEMINI_API_KEY') or os.getenv('GOOGLE_API_KEY'):
    available_backends.append('gemini')
if os.getenv('ANTHROPIC_API_KEY'):
    available_backends.append('anthropic')
if os.getenv('OPENAI_API_KEY'):
    available_backends.append('openai')

print(f"Available backends: {available_backends}")
assert len(available_backends) > 0, "At least one API key must be configured!"

## 4. Clone Repository & Set Up Paths

Clone the OMS repository to access skill/plugin YAML files and the data generation script.
If running locally within the repo, skip the clone step.

In [ ]:
# Check if we're already inside the repo
REPO_ROOT = Path('.').resolve()

# Walk up to find the repo root (look for content/skills/)
for parent in [REPO_ROOT] + list(REPO_ROOT.parents):
    if (parent / 'content' / 'skills').exists():
        REPO_ROOT = parent
        break
else:
    # Clone the repo if not found
    print("Cloning open-medical-skills repository...")
    !git clone --depth 1 https://github.com/Open-Medica/open-medical-skills.git /content/open-medical-skills
    REPO_ROOT = Path('/content/open-medical-skills')

# Add the scripts directory to Python path
SCRIPTS_DIR = REPO_ROOT / 'oms-models' / 'data' / 'scripts'
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

print(f"Repository root: {REPO_ROOT}")
print(f"Skills directory: {REPO_ROOT / 'content' / 'skills'}")
print(f"Plugins directory: {REPO_ROOT / 'content' / 'plugins'}")

## 5. Load Source Data

Load all three data sources and display statistics: counts per category, average description lengths, and source distribution.

In [ ]:
# Import loaders from the existing script
from generate_synthetic_queries import (
    load_oms_skills,
    load_oms_plugins,
    load_tu_tools,
    build_item_description,
    get_item_name,
    get_item_category,
    OMS_CATEGORIES,
    QUERY_TYPES,
)

# Load all data
skills = load_oms_skills()
plugins = load_oms_plugins()
tu_tools = load_tu_tools()

all_items = skills + plugins + tu_tools

print(f"\n{'='*60}")
print(f"DATA SUMMARY")
print(f"{'='*60}")
print(f"OMS Skills:        {len(skills):>6}")
print(f"OMS Plugins:       {len(plugins):>6}")
print(f"ToolUniverse Tools:{len(tu_tools):>6}")
print(f"{'─'*60}")
print(f"Total items:       {len(all_items):>6}")

In [ ]:
# Category distribution
categories = Counter(get_item_category(item) for item in all_items)
print(f"\n{'='*60}")
print(f"CATEGORY DISTRIBUTION")
print(f"{'='*60}")
for cat, count in categories.most_common():
    bar = '#' * min(count // 5, 40)
    print(f"  {cat:<35} {count:>5}  {bar}")

# Average description length
desc_lengths = [len(build_item_description(item)) for item in all_items]
print(f"\nDescription length stats:")
print(f"  Min:  {min(desc_lengths):>6} chars")
print(f"  Max:  {max(desc_lengths):>6} chars")
print(f"  Mean: {sum(desc_lengths)/len(desc_lengths):>6.0f} chars")

## 6. Augment Descriptions (Optional)

Enrich tool descriptions with clinical scenarios, synonym expansions, and contextual details.
This improves query diversity by giving the LLM richer context.

In [ ]:
def augment_description(item: dict) -> str:
    """Augment a tool description with clinical context and synonyms."""
    base_desc = build_item_description(item)
    source = item.get('_source', 'unknown')
    category = get_item_category(item)

    # Add clinical context based on category
    context_map = {
        'diagnosis': 'Used in diagnostic workup and differential diagnosis workflows.',
        'treatment': 'Assists with treatment planning, drug selection, and therapy management.',
        'lab-imaging': 'Supports laboratory test interpretation and medical imaging analysis.',
        'pharmacy': 'Helps with medication management, drug interactions, and formulary decisions.',
        'emergency': 'Designed for acute care, triage, and emergency department workflows.',
        'surgery': 'Assists with surgical planning, risk assessment, and perioperative care.',
        'nursing': 'Supports nursing assessment, care planning, and documentation.',
        'pediatrics': 'Tailored for pediatric patients with age-appropriate calculations and guidelines.',
        'mental-health': 'Assists with psychiatric assessment, screening, and treatment planning.',
        'public-health': 'Supports population health, epidemiology, and public health surveillance.',
        'research': 'Facilitates medical research, literature review, and evidence synthesis.',
        'education': 'Supports medical education, training, and competency assessment.',
        'administrative': 'Helps with healthcare administration, billing, and compliance.',
        'clinical-research-summarizing': 'Assists with clinical research summarization and evidence grading.',
    }

    augmented = base_desc
    if category in context_map:
        augmented += f"\nClinical context: {context_map[category]}"

    return augmented


# Show before/after examples
print("AUGMENTATION EXAMPLES")
print("=" * 60)
for item in all_items[:3]:
    name = get_item_name(item)
    print(f"\n--- {name} ({item.get('_source', '?')}) ---")
    print(f"\nBEFORE:\n{build_item_description(item)[:200]}...")
    print(f"\nAFTER:\n{augment_description(item)[:250]}...")
    print()

## 7. Generate Synthetic Queries

Call the LLM backend to generate 10-20 diverse queries per tool/skill.
This is the most time-consuming step -- expect ~2 hours for all 2,049 items with Gemini free tier.

The process supports **checkpointing**: if interrupted, re-run this cell to resume from where it left off.

In [ ]:
from generate_synthetic_queries import (
    get_backend,
    generate_queries_for_items,
    RateLimiter,
    compute_item_id,
    load_checkpoint,
)

# Configuration
BACKEND_NAME = 'gemini'  # 'gemini', 'openai', or 'anthropic'
QUERIES_PER_ITEM = 15    # 10-20 recommended
RATE_LIMIT_RPM = 15      # API calls per minute (Gemini free: 15, paid: 60)

# File paths
output_file = OUTPUT_DIR / f'synthetic_queries_all_{BACKEND_NAME}.jsonl'
checkpoint_file = OUTPUT_DIR / f'checkpoint_all_{BACKEND_NAME}.txt'

# Check progress
already_done = load_checkpoint(checkpoint_file)
remaining = [item for item in all_items if compute_item_id(item) not in already_done]

print(f"Backend:     {BACKEND_NAME}")
print(f"Total items: {len(all_items)}")
print(f"Completed:   {len(already_done)}")
print(f"Remaining:   {len(remaining)}")
print(f"Output:      {output_file}")
print(f"\nEstimated time: ~{len(remaining) * 60 / RATE_LIMIT_RPM / 60:.1f} hours")

In [ ]:
# Initialize backend and run generation
backend = get_backend(BACKEND_NAME)
rate_limiter = RateLimiter(calls_per_minute=RATE_LIMIT_RPM)

print(f"Starting query generation with {BACKEND_NAME}...")
print(f"Progress is checkpointed -- safe to interrupt and resume.\n")

total_written = generate_queries_for_items(
    items=all_items,
    backend=backend,
    output_path=output_file,
    checkpoint_path=checkpoint_file,
    queries_per_item=QUERIES_PER_ITEM,
    rate_limiter=rate_limiter,
)

print(f"\nGeneration complete! Total queries written this session: {total_written}")

## 8. Quality Check

Sample 20 generated (query, tool) pairs and display them for manual review.
Check that queries are natural, diverse, and relevant to the tool description.

In [ ]:
import random

# Load all generated records
records = []
if output_file.exists():
    with open(output_file, 'r') as f:
        for line in f:
            if line.strip():
                records.append(json.loads(line))

print(f"Total generated records: {len(records)}")

# Sample 20 for review
sample_size = min(20, len(records))
sample = random.sample(records, sample_size)

print(f"\n{'='*80}")
print(f"RANDOM SAMPLE ({sample_size} pairs)")
print(f"{'='*80}")

for i, rec in enumerate(sample, 1):
    print(f"\n--- Sample {i}/{sample_size} ---")
    print(f"  Tool:       {rec.get('tool_name', 'N/A')}")
    print(f"  Source:     {rec.get('source', 'N/A')}")
    print(f"  Category:   {rec.get('category', 'N/A')}")
    print(f"  Query Type: {rec.get('query_type', 'N/A')}")
    print(f"  Query:      {rec.get('query', 'N/A')}")
    desc = rec.get('tool_description', 'N/A')
    print(f"  Desc:       {desc[:120]}{'...' if len(desc) > 120 else ''}")

## 9. Save Output

Save the complete dataset in both JSONL (for easy streaming) and HuggingFace Dataset format (for training).

In [ ]:
from datasets import Dataset

# The JSONL file already exists from generation step.
# Now also save as HuggingFace Dataset for easy loading during training.

if records:
    ds = Dataset.from_list(records)
    hf_output_path = OUTPUT_DIR / 'synthetic_queries_hf'
    ds.save_to_disk(str(hf_output_path))
    print(f"HuggingFace Dataset saved to: {hf_output_path}")
    print(f"JSONL saved to: {output_file}")
    print(f"\nDataset info:")
    print(ds)
else:
    print("No records to save. Run the generation step first.")

## 10. Statistics & Distribution Analysis

Analyze the generated dataset: total pairs, distribution by source, category, and query type.

In [ ]:
if records:
    source_counts = Counter(r['source'] for r in records)
    category_counts = Counter(r['category'] for r in records)
    query_type_counts = Counter(r['query_type'] for r in records)

    print(f"{'='*60}")
    print(f"FINAL DATASET STATISTICS")
    print(f"{'='*60}")
    print(f"\nTotal pairs: {len(records)}")

    print(f"\n--- By Source ---")
    for src, count in source_counts.most_common():
        pct = 100 * count / len(records)
        print(f"  {src:<20} {count:>6} ({pct:5.1f}%)")

    print(f"\n--- By Category (top 15) ---")
    for cat, count in category_counts.most_common(15):
        pct = 100 * count / len(records)
        bar = '#' * int(pct)
        print(f"  {cat:<35} {count:>6} ({pct:5.1f}%) {bar}")

    print(f"\n--- By Query Type ---")
    for qt, count in query_type_counts.most_common():
        pct = 100 * count / len(records)
        bar = '#' * int(pct)
        print(f"  {qt:<25} {count:>6} ({pct:5.1f}%) {bar}")

    # Query length stats
    query_lengths = [len(r['query']) for r in records]
    print(f"\n--- Query Length ---")
    print(f"  Min:    {min(query_lengths):>4} chars")
    print(f"  Max:    {max(query_lengths):>4} chars")
    print(f"  Mean:   {sum(query_lengths)/len(query_lengths):>4.0f} chars")
    print(f"  Median: {sorted(query_lengths)[len(query_lengths)//2]:>4} chars")
else:
    print("No records available. Run generation first.")

---

## Next Steps

Proceed to **02_data_validation.ipynb** to validate the generated pairs using multi-model consensus scoring.

**Files produced by this notebook:**
- `data/raw/synthetic_queries_all_<backend>.jsonl` -- Raw JSONL output
- `data/raw/synthetic_queries_hf/` -- HuggingFace Dataset format
- `data/raw/checkpoint_all_<backend>.txt` -- Checkpoint file for resume